# Fahrzeugbewertungen von CarWale herunterladen

Dieses Notebook lädt Fahrzeugbewertungen für ausgewählte Automarken von **CarWale** herunter.
Dabei werden für mehrere Fahrzeugmodelle der Titel, der Kommentar und die Sternebewertung ausgelesen.
Anschließend werden die gesammelten Daten in einem Pandas-DataFrame dargestellt und als JSON-Datei gespeichert.


## 1. Benötigte Bibliotheken importieren

Für das Abrufen und Verarbeiten der Webseiten werden folgende Bibliotheken verwendet:

- `requests` zum Herunterladen der HTML-Seiten
- `BeautifulSoup` zum Analysieren der HTML-Struktur
- `pandas` zum Erstellen und Bearbeiten des DataFrames
- `json` zum Speichern der Daten im JSON-Format


In [1]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd


## 2. Links zu den Bewertungsseiten sammeln

Zuerst werden die gewünschten Automarken und die maximale Anzahl der Fahrzeugmodelle pro Marke festgelegt.
Danach wird die Übersichtsseite jeder Marke geöffnet. Aus dieser Seite werden die Links zu den einzelnen Fahrzeugmodellen extrahiert und zu Bewertungs-URLs ergänzt.


In [2]:
# Automarken, deren Bewertungen heruntergeladen werden sollen
marks = ["byd-cars", "volkswagen-cars"]

# Maximale Anzahl der Fahrzeugmodelle pro Marke
number_of_models = 4

# Liste zum Speichern der Bewertungsseiten
sub_urls = []

# Jede ausgewählte Automarke einzeln verarbeiten
for mark in marks:
    brand_url = f"https://www.carwale.com/{mark}"
    print("Markenseite:", brand_url)

    # HTML-Inhalt der Markenseite herunterladen
    response = requests.get(brand_url)

    # Bei einem fehlerhaften HTTP-Status mit der nächsten Marke fortfahren
    if response.status_code != 200:
        print("Fehler beim Abrufen der URL:", brand_url)
        continue

    # HTML-Dokument analysieren
    soup = BeautifulSoup(response.content, "html.parser")

    # Links zu den ersten Fahrzeugmodellen der Marke suchen
    cars = soup.find_all(
        "a",
        class_="o-C o-os",
        href=True,
        limit=number_of_models
    )

    # Für jedes Modell die zugehörige Bewertungs-URL erzeugen
    for car in cars:
        sub_urls.append({
            "mark": mark,
            "sub_url": f"https://www.carwale.com{car['href']}reviews/"
        })

# Gefundene Bewertungsseiten anzeigen
print(sub_urls)


Markenseite: https://www.carwale.com/byd-cars
Markenseite: https://www.carwale.com/volkswagen-cars
[{'mark': 'byd-cars', 'sub_url': 'https://www.carwale.com/byd-cars/atto-3/reviews/'}, {'mark': 'byd-cars', 'sub_url': 'https://www.carwale.com/byd-cars/seal/reviews/'}, {'mark': 'byd-cars', 'sub_url': 'https://www.carwale.com/byd-cars/sealion-7/reviews/'}, {'mark': 'byd-cars', 'sub_url': 'https://www.carwale.com/byd-cars/emax-7/reviews/'}, {'mark': 'volkswagen-cars', 'sub_url': 'https://www.carwale.com/volkswagen-cars/taigun/reviews/'}, {'mark': 'volkswagen-cars', 'sub_url': 'https://www.carwale.com/volkswagen-cars/virtus/reviews/'}, {'mark': 'volkswagen-cars', 'sub_url': 'https://www.carwale.com/volkswagen-cars/tayron/reviews/'}, {'mark': 'volkswagen-cars', 'sub_url': 'https://www.carwale.com/volkswagen-cars/golf-gti/reviews/'}]


## 3. Beispiel einer gespeicherten Bewertungs-URL

Mit der folgenden Zelle kann kontrolliert werden, ob die erzeugten Links korrekt aufgebaut sind.


In [3]:
# Zweiten Eintrag der URL-Liste als Beispiel anzeigen
sub_urls[1]


{'mark': 'byd-cars',
 'sub_url': 'https://www.carwale.com/byd-cars/seal/reviews/'}

## 4. Bewertungen von den einzelnen Seiten extrahieren

Für jedes Fahrzeugmodell werden maximal 20 Bewertungsseiten durchsucht.
Aus jeder Bewertungskarte werden folgende Informationen ausgelesen:

- Titel der Bewertung
- Kommentartext
- Anzahl der ausgefüllten Sterne
- zugehörige Automarke

Wenn eine Seite weniger als zehn Bewertungskarten enthält, wird davon ausgegangen, dass die letzte Seite erreicht wurde. Die Schleife für dieses Fahrzeugmodell wird dann beendet.


In [4]:
# Liste zum Speichern aller extrahierten Bewertungen
reviews_list = []

# Browser-Kennung, damit die Anfrage wie ein normaler Browser-Aufruf wirkt
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Alle zuvor erzeugten Bewertungs-URLs durchlaufen
for item in sub_urls:
    mark = item["mark"]
    base_url = item["sub_url"]

    # Pro Fahrzeugmodell maximal 20 Seiten durchsuchen
    for page_number in range(1, 21):
        review_url = f"{base_url}page/{page_number}"

        # Aktuelle Bewertungsseite herunterladen
        response = requests.get(review_url, headers=headers)

        # Bei einem HTTP-Fehler die aktuelle Modellschleife beenden
        if response.status_code != 200:
            print("Fehler beim Abrufen der URL:", review_url)
            break

        # HTML-Inhalt der Bewertungsseite analysieren
        soup = BeautifulSoup(response.content, "html.parser")

        # Alle Bewertungskarten auf der aktuellen Seite suchen
        cards = soup.find_all(
            "li",
            class_=lambda css_class: css_class and "oxygen-card-wrapper" in css_class
        )

        number_of_cards = len(cards)
        print(f"{mark} | Seite {page_number} | Anzahl der Bewertungen: {number_of_cards}")

        # Informationen aus jeder einzelnen Bewertungskarte extrahieren
        for card in cards:
            # Titel der Bewertung suchen
            title = card.find(
                "a",
                href=lambda href: href and "/reviews/" in href
            )

            # Text des Kommentars suchen
            comment = card.find(
                "div",
                attrs={"color": "dimGray"}
            )

            # Alle Sternsymbole innerhalb der Bewertung suchen
            stars = card.find_all(
                "svg",
                attrs={"aria-label": "rating icon"}
            )

            # Nur ausgefüllte Sterne anhand ihrer CSS-Klasse auswählen
            filled_stars = [
                star
                for star in stars
                if "o-k3" in star.get("class", [])
            ]

            # Anzahl der ausgefüllten Sterne entspricht der Bewertung
            rating = len(filled_stars)

            # Extrahierte Informationen als Dictionary speichern
            reviews_list.append({
                "Title": title.get_text(" ", strip=True) if title else None,
                "Comment": comment.get_text(" ", strip=True) if comment else None,
                "Rating": rating,
                "mark": mark
            })

        # Bei weniger als zehn Karten wurde wahrscheinlich die letzte Seite erreicht
        if number_of_cards < 10:
            break


byd-cars | Seite 1 | Anzahl der Bewertungen: 10
byd-cars | Seite 2 | Anzahl der Bewertungen: 10
byd-cars | Seite 3 | Anzahl der Bewertungen: 5
byd-cars | Seite 1 | Anzahl der Bewertungen: 10
byd-cars | Seite 2 | Anzahl der Bewertungen: 5
byd-cars | Seite 1 | Anzahl der Bewertungen: 9
byd-cars | Seite 1 | Anzahl der Bewertungen: 5
volkswagen-cars | Seite 1 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 2 | Anzahl der Bewertungen: 6
volkswagen-cars | Seite 1 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 2 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 3 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 4 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 5 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 6 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 7 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 8 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 9 | Anzahl der Bewertungen: 10
volkswagen-cars | Seite 10 | Anzahl der Bewertungen: 9
volkswage

## 5. Gesammelte Rohdaten kontrollieren

Die folgende Ausgabe dient zur ersten Kontrolle der extrahierten Datensätze.
Bei einer großen Datenmenge kann die Ausgabe sehr lang werden.


In [5]:
# Alle gesammelten Bewertungen anzeigen
print(reviews_list)


[{'Title': 'No sound,  smooth riding, comfortable. Best 5 seater electric Vehicle.', 'Comment': 'The driving experience is awesome. No sound,  smooth riding, comfortable. Best 5 seater. Nice display screen. Feelings awesome after driving this car. The build quality is good. The interior design is perfect.', 'Rating': 4, 'mark': 'byd-cars'}, {'Title': 'Dont Buy - After sales is awwful... you will be stuck', 'Comment': 'Me and my colleague bought BYD Atto 3 months ago. Its a wondeful car to drive and we were very happy unless the car met with an accident.It require change in back light and some work on truck. No internal damage was done. It has been over 4 weeks, BYD is unable to repair the car.They only have 1 reason, part is not available.I would recommend not to buy as you will be stuck for after sales support and parts.', 'Rating': 1, 'mark': 'byd-cars'}, {'Title': 'The BYD Atto 3 claims to offer a distance of 420km', 'Comment': 'The BYD Atto 3 claims to offer a distance of 420km I h

## 6. Pandas-DataFrame erstellen

Die Liste aus Dictionaries wird in einen DataFrame umgewandelt. Dadurch können die Daten einfacher kontrolliert, analysiert und später weiterverarbeitet werden.


In [6]:
# Gesammelte Bewertungen in einen DataFrame umwandeln
df = pd.DataFrame(reviews_list)

df


,Title,Comment,Rating,mark
0,"No sound, smooth riding, comfortable. Best 5 ...","The driving experience is awesome. No sound, ...",4,byd-cars
1,Dont Buy - After sales is awwful... you will b...,Me and my colleague bought BYD Atto 3 months a...,1,byd-cars
2,The BYD Atto 3 claims to offer a distance of 4...,The BYD Atto 3 claims to offer a distance of 4...,1,byd-cars
3,DO NOT BUY BYD CAR PLEASE READ THE COMMENTS AN...,"Worst is better word for me to use here,car pe...",1,byd-cars
4,Car is good but after service is very bad,If there’s an opportunity to provide feedback ...,1,byd-cars
...,...,...,...,...
169,Volkswagen Tayron,The leather Seats on the Kodiaq are nice but o...,5,volkswagen-cars
170,"Perfect blend of Comfort, Performance, Appeare...",I bought this vehicle on 01 June. It's a whole...,5,volkswagen-cars
171,Without Taxes It Would Be A Fun Ride,"With that headline, I meant to say that this c...",5,volkswagen-cars
172,Best car,"I drove this car ,best in performance, It has ...",5,volkswagen-cars


## 7. Daten als JSON-Datei speichern

Zum Speichern wird der DataFrame zunächst wieder in eine Liste von Dictionaries umgewandelt.
Die Option `ensure_ascii=False` sorgt dafür, dass Sonderzeichen lesbar gespeichert werden. Mit `indent=4` wird die JSON-Datei übersichtlich formatiert.


In [7]:
# DataFrame in eine Liste von Dictionaries umwandeln
data = df.to_dict(orient="records")

# Bewertungen als formatierte JSON-Datei speichern
with open("../data/cars_reviews.json", "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=4)


## Ergebnis

Die heruntergeladenen Fahrzeugbewertungen befinden sich nun in der Datei `cars_reviews.json`.
Jeder Datensatz enthält den Bewertungstitel, den Kommentar, die Sternebewertung und die Automarke.
